# Stage 3: main training run (Qwen3-1.7B + 10,000 Urdu tokens)

Continued pretraining of the extended model: first only the new embedding rows, then LoRA plus the new rows, on Urdu Wikipedia with 10% English mixed in. Notebooks 05 (smoke) and 06 (speed) must have passed first.

**Built to be safe, not fast to finish:**

- A checkpoint goes to your private Hub repo every **50 steps** (`<repo>/last`), and every **200th** is kept for good in `<repo>/milestones/step-NNNNNNN`.
- If the session ends (12-hour limit, crash, disconnect), run this notebook again with the **same settings**. It finds `last/` on the Hub and resumes from there, with the same data order.
- Every 100 steps it measures held-out Urdu bits per character and English bits per character, so you can see learning and any English damage as it happens.
- Any milestone can be merged and evaluated later, e.g. `zubaan adapt merge --config adapt.yaml --milestone 400`. If you stop early, merge the newest checkpoint: usable, but its learning rate has not decayed, so it is a little noisier than a finished run.

Settings before running: Accelerator **GPU T4 x2**, Internet **on**, Input **Add Input > Notebooks** with the output of `03_extend_export_qwen3`, Secret **HF_TOKEN** attached. Use *Save Version > Save & Run All (Commit)* so it keeps running with the tab closed. Set `HUB_REPO` and the speed-trial winner in the first cell.

In [ ]:
# The Hub repo is required: it is what makes the run safe. Create it as PRIVATE first. It must be a
# different repo from the smoke one (notebook 05 used HUB_REPO + "-smoke", so there is no clash).
HUB_REPO = "hashiraziz21/zubaan-qwen3-ckpt"            # e.g. "your-name/zubaan-qwen3-ckpt"
MICRO_BATCH = 1          # from notebook 06 (T4, 2026-11-18): 850 tok/s, peak 13.3 GB; micro batch 2 without checkpointing ran out of memory
GRAD_CHECKPOINT = False  # from notebook 06: off is 44% faster than on (592 tok/s) and still fits
PHASE_A_STEPS = 100      # steps that train only the 10,000 new rows
PHASE_B_STEPS = 700      # steps that train LoRA plus the new rows; the learning rate decays to its minimum over these
# Do not change any of the lines above between sessions of the same run: a resumed run refuses a changed schedule.
import os

# Peak memory is 13.3 GB of about 14.5: let the allocator avoid fragmentation. Must be set before torch starts.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
assert HUB_REPO, "set HUB_REPO: without it a crash or the 12-hour limit loses the run"
assert 32 % MICRO_BATCH == 0, "MICRO_BATCH must divide 32"
OUT_NAME = "adapted"
HUB = HUB_REPO
MODE = "main"

In [ ]:
# Install zubaan at one exact commit. Do not edit the next line except to change the commit.
COMMIT = "98fb1a5"
REPO = f"git+https://github.com/Hashir-Aziz123/zubaan.git@{COMMIT}"
!pip install -q "zubaan[audit,train] @ {REPO}"
# Kaggle may already hold an older copy with the same version number, which pip keeps.
# Force our commit (dependencies are already in place).
!pip install -q --force-reinstall --no-deps "zubaan @ {REPO}"
# Kaggle ships torchao 0.10, which peft refuses ("only versions above 0.16 are supported") when it
# builds LoRA layers. We never use torchao, so remove it; peft then skips its torchao support.
!pip uninstall -y -q torchao
!zubaan version

In [ ]:
# CHECK 1: the installed code is the commit above, not a cached older copy.
# If this fails, use Run > Restart session and run the cells again from the top.
import json
from importlib import metadata

direct = json.loads(metadata.distribution("zubaan").read_text("direct_url.json"))
installed = direct["vcs_info"]["commit_id"]
print("installed commit:", installed)
assert installed.startswith(COMMIT), f"installed {installed}, wanted {COMMIT}"

import zubaan.adapt.pipeline  # noqa: F401

print("OK: code version verified")

In [ ]:
import os
import subprocess

# Quiet progress bars: they can overwhelm the notebook front end.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

# CHECK 2: a GPU with about 16 GB. Training uses one GPU (fp16: a T4 and a P100 have no BF16).
try:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
except FileNotFoundError:
    raise SystemExit("No GPU in this session (nvidia-smi is missing). Settings > Accelerator > GPU T4 x2, then run again.") from None
assert r.returncode == 0, "no GPU found: turn on a GPU accelerator in the notebook settings"
print(r.stdout)
gpu_name, cap, mem = [x.strip() for x in r.stdout.splitlines()[0].split(",")]
assert float(mem.split()[0]) >= 14000, f"{gpu_name} has {mem}; need about 16 GB"
if float(cap) < 7.0:
    print("WARNING: compute capability", cap, "has no fp16 tensor cores; this will be slow. A T4 is better.")

import peft
import torch
import transformers

print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__, "| cuda", torch.version.cuda)
assert torch.cuda.is_available(), "torch cannot see the GPU"
print("OK: GPU", gpu_name, "compute capability", cap, mem)

In [ ]:
# The Hub token is a Kaggle secret named HF_TOKEN (Add-ons > Secrets). It is never written in code.
if HUB_REPO:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("Hub checkpoints go to the PRIVATE repo", HUB_REPO, "(smoke runs use", HUB_REPO + "-smoke)")
else:
    print("No Hub repo: checkpoints stay in this session. A crash or the 12-hour limit loses the run.")

## 1. Find the extended model

In [ ]:
from pathlib import Path

# The extended model comes from the output of notebook 03, added as an input with "Add Input > Notebooks".
# The folder holds zubaan_extension.json. Prefer one that is not trained yet.
INPUT_ROOT = Path("/kaggle/input")
markers = sorted(INPUT_ROOT.rglob("zubaan_extension.json"))
assert markers, (
    "no extended model found under /kaggle/input. In the sidebar choose Add Input > Notebooks and add "
    "the finished output of 03_extend_export_qwen3"
)
untrained = [m for m in markers if not json.loads(m.read_text()).get("trained")]
marker = (untrained or markers)[0]
MODEL_DIR = marker.parent
print("model folder:", MODEL_DIR)
print(json.dumps(json.loads(marker.read_text()), indent=1))
assert (MODEL_DIR / "model.safetensors").exists() or list(MODEL_DIR.glob("model-*.safetensors")), "no weights in the model folder"
assert (MODEL_DIR / "tokenizer.json").exists(), "no tokenizer.json in the model folder"

## 2. Configuration

In [ ]:
from pathlib import Path

import yaml

WORK = Path("/kaggle/working")
adapt = {
    "model_path": str(MODEL_DIR),
    "urdu_corpus": "data/urdu/urdu_train.txt",
    "heldout": "data/urdu/urdu_heldout.txt",
    "english_corpus": "data/english/english_replay.txt",
    "english_mb": 12,
    "guard_urdu": ["data/flores_urd.txt", "data/flores_dev/flores_urd.txt"],
    "guard_english": ["data/flores_eng.txt", "data/flores_dev/flores_eng.txt"],
    "guard_belebele": True,
    "drop_overlaps": True,
    "data_dir": "data/adapt",
    "block_size": 1024,
    "english_replay_ratio": 0.1,
    "output_dir": str(WORK / OUT_NAME),
    "micro_batch_size": MICRO_BATCH,
    "grad_accum": 32 // MICRO_BATCH,
    "lr_new_rows": 1e-3,
    "lr_lora": 2e-4,
    "lora_rank": 64,
    "lora_alpha": 128,
    "lora_dropout": 0.05,
    "precision": "fp16",
    "gradient_checkpointing": GRAD_CHECKPOINT,
    "device": "cuda",
    "seed": 0,
    "hub_repo": HUB,
    "hub_private": True,
    "merged_dtype": "bfloat16",
    "phase_a_steps": PHASE_A_STEPS,
    "phase_b_steps": PHASE_B_STEPS,
    "warmup_steps": 50,
    "log_every": 10,
    "eval_every": 100,
    "eval_texts": 200,
    "checkpoint_every": 50,
    "keep_checkpoints": 2,
    "milestone_every": 200,
}
config = {
    "urdu_corpus": {"raw_dir": "data/raw", "out_dir": "data/urdu", "flores": "data/flores_urd.txt", "max_mb": 100, "heldout_percent": 1.0},
    "adapt": adapt,
}
Path("adapt.yaml").write_text(yaml.safe_dump(config, sort_keys=False))
print(Path("adapt.yaml").read_text())

## 3. Data

The same preparation as notebook 05 (about 10 minutes).

In [ ]:
# Evaluation text first (FLORES devtest and dev), then the Urdu training text, then the token blocks.
# Prepare drops any training line that overlaps FLORES or Belebele text and counts it.
!zubaan data flores --out data
!zubaan data flores --split dev --out data/flores_dev
!zubaan data urdu --config adapt.yaml
!zubaan adapt prepare --config adapt.yaml

In [ ]:
meta = json.loads(Path("data/adapt/blocks_meta.json").read_text())
print("Urdu blocks", meta["urdu"]["n_blocks"], "| tokens", meta["urdu"]["n_tokens"], "| English blocks", meta["english"]["n_blocks"])
print("guard:", meta["overlap_guard"])
assert meta["overlap_guard"]["hits"] == 0
# The laptop run (2026-11-18) gave 15,677 Urdu blocks and 16,053,605 tokens. A different number means a
# different Wikipedia dump or tokenizer: not wrong by itself, but write it down.
print("laptop reference: 15677 Urdu blocks, 16053605 tokens")

## 4. Hub check

Proves the repo is private and writable before hours are spent, and says whether this run starts fresh or resumes.

In [ ]:
# Before hours of training: the repo exists, is private, accepts a write, and tells us where we start.
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(HUB_REPO, private=True, exist_ok=True)
assert api.repo_info(HUB_REPO).private, f"{HUB_REPO} is public: make it private on the Hub first"
api.upload_file(path_or_fileobj=b"ok", path_in_repo="_preflight/ok.txt", repo_id=HUB_REPO)
api.delete_folder("_preflight", repo_id=HUB_REPO)
files = set(api.list_repo_files(HUB_REPO))
if "last/meta.json" in files:
    from huggingface_hub import hf_hub_download

    step = json.loads(Path(hf_hub_download(HUB_REPO, "last/meta.json")).read_text())["step"]
    print(f"OK: Hub writable. A checkpoint at step {step} exists: this run RESUMES from it.")
else:
    print("OK: Hub writable. No checkpoint yet: this run starts from step 0.")
print("milestones already saved:", sorted({f.split("/")[1] for f in files if f.startswith("milestones/")}))

## 5. Train

Run it again with the same settings if the session ends before it prints `finished`.

In [ ]:
from zubaan.adapt.pipeline import run_adapt
from zubaan.config import load_config

cfg = load_config("adapt.yaml").adapt
summary = run_adapt(cfg, log=print)
print({k: summary[k] for k in ("resumed_from_step", "final_step", "total_steps", "finished")})
assert summary["finished"], "the schedule did not finish: run this notebook again with the same settings to resume"
for e in summary["evals"]:
    print(e["step"], e["eval"])

## 6. Merge, save and verify

In [ ]:
# Fold LoRA and the trained rows into the weights (in float32), save a plain model with the chat
# template files, and verify the folder. Needs the run to have finished.
!zubaan adapt merge --config adapt.yaml

## 7. Output size

In [ ]:
total = 0
for f in sorted(WORK.rglob("*")):
    if f.is_file() and f.stat().st_size > 5e6:
        total += f.stat().st_size
        print(f"{f.stat().st_size / 1e9:6.2f} GB  {f.relative_to(WORK)}")
print(f"total of files over 5 MB: {total / 1e9:.1f} GB")
assert total < 19e9, "output is close to Kaggle's 20 GB limit: delete data/raw or old checkpoints"